# Water Quality Data Preprocessing and Feature Engineering

# Setup and Library Imports

In [1]:
import os
import sys
import json
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Project root directory resolution
ROOT_DIR = '.'
for candidate in ['.', '..', '../..']:
    if os.path.exists(os.path.join(candidate, 'data', 'Dataset')):
        ROOT_DIR = os.path.abspath(candidate)
        break

print('Project Root:', ROOT_DIR)


Project Root: /Users/mayanksharma/Downloads/New_Projects/Semester_5_AML_DL


# Feature Definitions and Mapping Constants

In [2]:
COLUMN_NAME_MAP = {
    'Country': 'country',
    'Area': 'area',
    'Waterbody Type': 'waterbody_type',
    'Date': 'date',
    'Ammonia (mg/l)': 'ammonia_mg_l',
    'Biochemical Oxygen Demand (mg/l)': 'bod_mg_l',
    'Dissolved Oxygen (mg/l)': 'do_mg_l',
    'Orthophosphate (mg/l)': 'orthophosphate_mg_l',
    'pH (ph units)': 'ph',
    'Temperature (cel)': 'temperature_c',
    'Nitrogen (mg/l)': 'nitrogen_mg_l',
    'Nitrate (mg/l)': 'nitrate_mg_l',
    'CCME_Values': 'ccme_values',
    'CCME_WQI': 'ccme_wqi',
}

SENTINEL_VALUES = [-999999.0, -9999.0, -999.0, -596.0, -99.9, -80.0]
SENTINEL_COLUMNS = ['ph', 'temperature_c', 'do_mg_l']

NEGATIVE_CLEANING_RULES = {
    'ammonia_mg_l': -0.01,
    'bod_mg_l': -0.10,
    'do_mg_l': -0.15,
    'orthophosphate_mg_l': -0.01,
    'nitrogen_mg_l': -0.01,
    'nitrate_mg_l': -0.01,
}

CCME_WQI_CATEGORIES = ['Poor', 'Marginal', 'Fair', 'Good', 'Excellent']
CCME_WQI_MAP = {cat: idx for idx, cat in enumerate(CCME_WQI_CATEGORIES)}

BASE_NUMERIC_FEATURES = [
    'ph',
    'temperature_c',
    'do_mg_l',
    'bod_mg_l',
    'ammonia_mg_l',
    'orthophosphate_mg_l',
    'nitrogen_mg_l',
    'nitrate_mg_l',
]

ENGINEERED_FEATURES = [
    'np_ratio',
    'bod_do_ratio',
    'do_deficit',
    'month',
    'quarter',
    'month_sin',
    'month_cos',
]

MODEL_INPUT_FEATURES = BASE_NUMERIC_FEATURES + ENGINEERED_FEATURES
TARGET_REGRESSION = 'ccme_values'
TARGET_CLASSIFICATION = 'ccme_wqi'
TARGET_CLASSIFICATION_ORDINAL = 'ccme_wqi_code'

print('Base Features (8):', BASE_NUMERIC_FEATURES)
print('Engineered Features (7):', ENGINEERED_FEATURES)
print('Total Modeling Features (15):', len(MODEL_INPUT_FEATURES))


Base Features (8): ['ph', 'temperature_c', 'do_mg_l', 'bod_mg_l', 'ammonia_mg_l', 'orthophosphate_mg_l', 'nitrogen_mg_l', 'nitrate_mg_l']
Engineered Features (7): ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']
Total Modeling Features (15): 15


# Core Preprocessing Functions

In [3]:
def clean_water_quality_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Performs full dataset cleaning, sentinel removal, and deduplication."""
    out = df.copy()

    # 1. Standardise column names
    existing_map = {k: v for k, v in COLUMN_NAME_MAP.items() if k in out.columns}
    out = out.rename(columns=existing_map)

    # 2. Cast data types
    if 'date' in out.columns:
        out['date'] = pd.to_datetime(out['date'], dayfirst=True, errors='coerce')

    if 'ccme_wqi' in out.columns:
        present_cats = [c for c in CCME_WQI_CATEGORIES if c in out['ccme_wqi'].values]
        cat_dtype = pd.CategoricalDtype(categories=present_cats, ordered=True)
        out['ccme_wqi'] = out['ccme_wqi'].astype(cat_dtype)

    null_strings = {'nan', 'null', 'none', 'na', 'n/a', 'nil', '', '?'}
    for col in ['country', 'area', 'waterbody_type']:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip()
            out.loc[out[col].str.lower().isin(null_strings), col] = np.nan

    # 3. Replace sentinels & scale anomalies
    for col in SENTINEL_COLUMNS:
        if col in out.columns:
            mask = (out[col] <= -999.0) | out[col].isin(SENTINEL_VALUES)
            out.loc[mask, col] = np.nan

    if 'ph' in out.columns:
        out.loc[(out['ph'] < 0.0) | (out['ph'] > 14.0), 'ph'] = np.nan

    if 'temperature_c' in out.columns:
        out.loc[out['temperature_c'] < -10.0, 'temperature_c'] = np.nan

    # 4. Fix historical USA temperatures recorded in Fahrenheit (T > 50 degF)
    if 'temperature_c' in out.columns and 'country' in out.columns:
        mask_f = (out['country'] == 'USA') & (out['temperature_c'] > 50.0)
        out.loc[mask_f, 'temperature_c'] = (out.loc[mask_f, 'temperature_c'] - 32.0) * 5.0 / 9.0

    # 5. Clean negative concentrations (detection limits clipped to 0.0)
    for col, noise_min in NEGATIVE_CLEANING_RULES.items():
        if col in out.columns:
            out.loc[(out[col] >= noise_min) & (out[col] < 0.0), col] = 0.0
            out.loc[out[col] < noise_min, col] = np.nan

    # 6. Deduplicate exact duplicate records
    out = out.drop_duplicates()
    return out


def calculate_do_saturation(temp_c: pd.Series) -> pd.Series:
    """Calculates theoretical freshwater dissolved oxygen saturation (mg/L) at 1 atm via Elmore & Hayes."""
    is_na = temp_c.isna()
    t = temp_c.clip(lower=0.0, upper=45.0)
    sat = 14.652 - (0.41022 * t) + (0.007991 * (t ** 2)) - (0.000077774 * (t ** 3))
    return sat.where(~is_na, np.nan)


def engineer_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Computes nutrient ratios, oxygen deficit, cyclical seasonal features, and ordinal targets."""
    out = df.copy()
    new_features = []

    # Stoichiometric N:P ratio
    if 'nitrogen_mg_l' in out.columns and 'orthophosphate_mg_l' in out.columns:
        valid_np = (
            out['nitrogen_mg_l'].notna()
            & out['orthophosphate_mg_l'].notna()
            & (out['orthophosphate_mg_l'] > 0.0)
            & (out['nitrogen_mg_l'] >= 0.0)
        )
        out['np_ratio'] = np.where(valid_np, out['nitrogen_mg_l'] / out['orthophosphate_mg_l'], np.nan)
        new_features.append('np_ratio')

    # Organic loading BOD:DO ratio
    if 'bod_mg_l' in out.columns and 'do_mg_l' in out.columns:
        valid_bod_do = (
            out['bod_mg_l'].notna()
            & out['do_mg_l'].notna()
            & (out['do_mg_l'] > 0.0)
            & (out['bod_mg_l'] >= 0.0)
        )
        out['bod_do_ratio'] = np.where(valid_bod_do, out['bod_mg_l'] / out['do_mg_l'], np.nan)
        new_features.append('bod_do_ratio')

    # Oxygen deficit
    if 'do_mg_l' in out.columns and 'temperature_c' in out.columns:
        valid_d = out['do_mg_l'].notna() & out['temperature_c'].notna()
        sat = calculate_do_saturation(out['temperature_c'])
        out['do_deficit'] = np.where(valid_d, sat - out['do_mg_l'], np.nan)
        new_features.append('do_deficit')

    # Cyclical seasonal features
    if 'date' in out.columns and pd.api.types.is_datetime64_any_dtype(out['date']):
        month = out['date'].dt.month
        out['month'] = month
        out['quarter'] = out['date'].dt.quarter
        out['month_sin'] = np.where(month.notna(), np.sin(2.0 * np.pi * (month - 1.0) / 12.0), np.nan)
        out['month_cos'] = np.where(month.notna(), np.cos(2.0 * np.pi * (month - 1.0) / 12.0), np.nan)
        new_features.extend(['month', 'quarter', 'month_sin', 'month_cos'])

    # Ordinal target code
    if 'ccme_wqi' in out.columns:
        out['ccme_wqi_code'] = out['ccme_wqi'].map(CCME_WQI_MAP).astype('Int64')

    return out, new_features


def split_data(
    df: pd.DataFrame,
    train_size: float = 0.70,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Performs stratified 70/15/15 split strictly preventing data leakage."""
    val_test_ratio = val_size + test_size
    stratify_col = 'ccme_wqi' if 'ccme_wqi' in df.columns else None

    train_df, holdout_df = train_test_split(
        df,
        test_size=val_test_ratio,
        random_state=random_state,
        stratify=df[stratify_col] if stratify_col else None,
    )

    test_ratio_of_holdout = test_size / val_test_ratio
    val_df, test_df = train_test_split(
        holdout_df,
        test_size=test_ratio_of_holdout,
        random_state=random_state,
        stratify=holdout_df[stratify_col] if stratify_col else None,
    )
    return train_df.copy(), val_df.copy(), test_df.copy()


def impute_missing_values(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """Fits domain-informed median imputation strictly on the training set and transforms all splits."""
    train_out = train_df.copy()
    val_out = val_df.copy()
    test_out = test_df.copy()

    global_medians = {
        col: float(train_df[col].median(skipna=True))
        for col in feature_cols
        if col in train_df.columns and pd.api.types.is_numeric_dtype(train_df[col])
    }

    country_medians = {}
    if 'country' in train_df.columns:
        for col in feature_cols:
            if col in train_df.columns and pd.api.types.is_numeric_dtype(train_df[col]):
                country_medians[col] = train_df.groupby('country', observed=False)[col].median().to_dict()

    chem_group_cols = [c for c in ['country', 'waterbody_type'] if c in train_df.columns]
    temp_group_cols = [c for c in ['country', 'month'] if c in train_df.columns]

    def _get_key(d, cols):
        if len(cols) == 2:
            return d[cols[0]].astype(str) + '___' + d[cols[1]].astype(str)
        elif len(cols) == 1:
            return d[cols[0]].astype(str)
        return pd.Series('', index=d.index)

    chem_tr = _get_key(train_df, chem_group_cols)
    chem_val = _get_key(val_df, chem_group_cols)
    chem_te = _get_key(test_df, chem_group_cols)

    temp_tr = _get_key(train_df, temp_group_cols)
    temp_val = _get_key(val_df, temp_group_cols)
    temp_te = _get_key(test_df, temp_group_cols)

    seasonal_set = {'temperature_c', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos'}

    for col in feature_cols:
        if col not in train_out.columns or not pd.api.types.is_numeric_dtype(train_out[col]):
            continue

        g_med = global_medians.get(col, 0.0)
        c_med = country_medians.get(col, {})
        is_season = col in seasonal_set and len(temp_group_cols) > 0

        keys_tr, keys_val, keys_te = (temp_tr, temp_val, temp_te) if is_season else (chem_tr, chem_val, chem_te)
        group_meds = train_df.groupby(keys_tr, observed=False)[col].median().to_dict()

        for partition_df, keys in [(train_out, keys_tr), (val_out, keys_val), (test_out, keys_te)]:
            partition_df[col] = partition_df[col].fillna(keys.map(group_meds))
            if 'country' in partition_df.columns and c_med:
                partition_df[col] = partition_df[col].fillna(partition_df['country'].map(c_med))
            partition_df[col] = partition_df[col].fillna(g_med)

    imputer_info = {'global_medians': global_medians, 'feature_cols': feature_cols}
    return train_out, val_out, test_out, imputer_info


def scale_features(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, StandardScaler]:
    """Fits StandardScaler strictly on training input features and transforms train, val, and test."""
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])

    train_out = train_df.copy()
    val_out = val_df.copy()
    test_out = test_df.copy()

    train_out[feature_cols] = scaler.transform(train_df[feature_cols])
    val_out[feature_cols] = scaler.transform(val_df[feature_cols])
    test_out[feature_cols] = scaler.transform(test_df[feature_cols])

    return train_out, val_out, test_out, scaler


# Load and Clean Dataset

In [4]:
raw_path = os.path.join(ROOT_DIR, 'data', 'Dataset', 'Parquet', 'Combined Data', 'Combined_dataset.parquet')
cleaned_path = os.path.join(ROOT_DIR, 'data', 'GeneratedOutput', '02.Preprocessing&FeatureSelection', 'cleaned_water_quality.parquet')

if os.path.exists(cleaned_path):
    print(f'Loading existing cleaned dataset from: {cleaned_path}')
    df_clean = pd.read_parquet(cleaned_path)
else:
    print(f'Loading raw dataset and running cleaning pipeline...')
    df_raw = pd.read_parquet(raw_path)
    df_clean = clean_water_quality_dataset(df_raw)

print(f'Cleaned Dataset Dimensions: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')
df_clean[['country', 'area', 'waterbody_type', 'date', 'ph', 'temperature_c', 'do_mg_l', 'ccme_values', 'ccme_wqi']].head()


Loading raw dataset and running cleaning pipeline...


Cleaned Dataset Dimensions: 4,287,890 rows x 14 columns


,country,area,waterbody_type,date,ph,temperature_c,do_mg_l,ccme_values,ccme_wqi
0,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2014-12-01,NaN,24.7,4.15,93.102127,Good
1,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-02-02,NaN,22.4,12.07,100.000000,Excellent
2,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-03-02,NaN,17.0,10.59,100.000000,Excellent
3,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-04-13,7.4,17.0,3.42,100.000000,Excellent
4,Australia,Torrens River Adelaide Zoo Pedestrian Bridge,River,2015-01-15,7.4,23.2,3.91,92.814413,Good


# Feature Engineering

In [5]:
df_feat, new_cols = engineer_features(df_clean)
print('Engineered Features:', new_cols)
df_feat[['np_ratio', 'bod_do_ratio', 'do_deficit', 'month_sin', 'month_cos', 'ccme_wqi_code']].describe().round(4)


Engineered Features: ['np_ratio', 'bod_do_ratio', 'do_deficit', 'month', 'quarter', 'month_sin', 'month_cos']


,np_ratio,bod_do_ratio,do_deficit,month_sin,month_cos,ccme_wqi_code
count,2.817393e+06,2.821499e+06,3.140827e+06,4.287890e+06,4.287890e+06,4287890.0
mean,6.016730e+01,1.918900e+01,9.480000e-01,-3.500000e-03,-3.000000e-02,3.1843
std,3.630478e+02,3.953171e+03,1.941900e+00,7.105000e-01,7.031000e-01,1.0555
min,0.000000e+00,0.000000e+00,-1.691930e+01,-1.000000e+00,-1.000000e+00,0.0
25%,1.000000e+01,1.569000e-01,-3.750000e-02,-8.660000e-01,-8.660000e-01,3.0
50%,2.613640e+01,2.348000e-01,6.833000e-01,0.000000e+00,-0.000000e+00,4.0
75%,4.980390e+01,3.167000e-01,1.528600e+00,8.660000e-01,5.000000e-01,4.0
max,1.490000e+05,4.210000e+06,1.465200e+01,1.000000e+00,1.000000e+00,4.0


# Feature Selection and Target Quarantine

In [6]:
print('Model Input Features (15):')
for i, f in enumerate(MODEL_INPUT_FEATURES, 1):
    print(f'{i:2d}. {f}')

for target_col in [TARGET_REGRESSION, TARGET_CLASSIFICATION, TARGET_CLASSIFICATION_ORDINAL]:
    assert target_col not in MODEL_INPUT_FEATURES
print('Target quarantine check passed: targets are strictly isolated from model inputs.')

Model Input Features (15):
 1. ph
 2. temperature_c
 3. do_mg_l
 4. bod_mg_l
 5. ammonia_mg_l
 6. orthophosphate_mg_l
 7. nitrogen_mg_l
 8. nitrate_mg_l
 9. np_ratio
10. bod_do_ratio
11. do_deficit
12. month
13. quarter
14. month_sin
15. month_cos
Target quarantine check passed: targets are strictly isolated from model inputs.


# Split Dataset into Train, Validation, and Test Sets

In [7]:
train_raw, val_raw, test_raw = split_data(
    df_feat,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
)
print('Train Partition Rows (70%):', f'{len(train_raw):,}')
print('Val Partition Rows   (15%):', f'{len(val_raw):,}')
print('Test Partition Rows  (15%):', f'{len(test_raw):,}')


Train Partition Rows (70%): 3,001,523
Val Partition Rows   (15%): 643,183
Test Partition Rows  (15%): 643,184


# Impute Missing Values Using Training Set Medians

In [8]:
train_imp, val_imp, test_imp, imputer_info = impute_missing_values(
    train_raw, val_raw, test_raw, feature_cols=MODEL_INPUT_FEATURES
)
print('Remaining Missing Values in Modeling Features:')
print('Train:', int(train_imp[MODEL_INPUT_FEATURES].isna().sum().sum()))
print('Val:  ', int(val_imp[MODEL_INPUT_FEATURES].isna().sum().sum()))
print('Test: ', int(test_imp[MODEL_INPUT_FEATURES].isna().sum().sum()))


Remaining Missing Values in Modeling Features:
Train: 0
Val:   0
Test:  0


# Scale Modeling Features Using Training Statistics

In [9]:
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_imp, val_imp, test_imp, feature_cols=MODEL_INPUT_FEATURES
)
train_means = train_scaled[MODEL_INPUT_FEATURES].mean().abs()
train_stds = train_scaled[MODEL_INPUT_FEATURES].std(ddof=0)
print('Maximum mean deviation from 0.0:', f'{train_means.max():.6e}')
print('Maximum std deviation from 1.0: ', f'{abs(train_stds - 1.0).max():.6e}')


Maximum mean deviation from 0.0: 1.316204e-15
Maximum std deviation from 1.0:  2.220446e-16


# Validate Data Quality and Schema Parity

In [10]:
assert list(train_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(val_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(test_imp[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(train_scaled[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES

assert train_imp['ccme_values'].notna().all()
assert val_imp['ccme_values'].notna().all()
assert test_imp['ccme_values'].notna().all()
assert train_imp['ccme_wqi_code'].notna().all()
print('All data quality and schema validations passed successfully.')


All data quality and schema validations passed successfully.


# Verify Preprocessing Artifacts and Metadata

In [11]:
meta_path = os.path.join(ROOT_DIR, 'data', 'GeneratedOutput', '02.Preprocessing&FeatureSelection', 'preprocessing_metadata.json')
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)

    print('Metadata Version:    ', meta.get('metadata_version', 'N/A'))
    print('Total Master Records:', f"{meta.get('original_row_count', 0):,}")
    print('Cleaned Records:     ', f"{meta.get('cleaned_row_count', 0):,}")
    print('Train Records:       ', f"{meta.get('split_counts', {}).get('train', 0):,}")
    print('Validation Records:  ', f"{meta.get('split_counts', {}).get('val', 0):,}")
    print('Test Records:        ', f"{meta.get('split_counts', {}).get('test', 0):,}")
    print('Input Feature Count: ', meta.get('input_feature_count', 15))
else:
    print('Metadata file not yet generated at:', meta_path)


Metadata Version:     2.0.0
Total Master Records: 4,297,895
Cleaned Records:      4,287,890
Train Records:        3,001,523
Validation Records:   643,183
Test Records:         643,184
Input Feature Count:  15
